# C2 — Déboguer avec le logger

**Objectif** : comprendre comment un fichier de log permet de tracer et déboguer un programme.

Ce notebook simule le comportement de `jeu/logger.py` de Drone Rescue en pur Python (compatible Google Colab).

## Étape 1 — Logger minimal : ouvrir, écrire, fermer

In [ ]:
import io

# Simulation en mémoire (pas de fichier disque sur Colab)
log_buffer = io.StringIO()

def demarrer_log(buf):
    buf.seek(0)
    buf.truncate()
    buf.write("=== DRONE RESCUE — Journal de partie ===\n")
    buf.write("Tour | Entité | Mouvement | Événement\n")
    buf.write("-" * 50 + "\n")

def enregistrer_log(buf, ligne):
    buf.write(ligne + "\n")

# Test
demarrer_log(log_buffer)
enregistrer_log(log_buffer, "Tour 1 | drone_1 | N | déplacement OK")
enregistrer_log(log_buffer, "Tour 1 | drone_1 | — | survivant S1 sauvé")
print(log_buffer.getvalue())

## Étape 2 — Ajouter un horodatage

In [ ]:
from datetime import datetime

def enregistrer_log_horodatage(buf, ligne):
    horodatage = datetime.now().strftime("%H:%M:%S")
    buf.write(f"[{horodatage}] {ligne}\n")

demarrer_log(log_buffer)
enregistrer_log_horodatage(log_buffer, "Tour 1 | drone_1 | N | déplacement OK")
enregistrer_log_horodatage(log_buffer, "Tour 2 | drone_2 | E | batterie faible")
print(log_buffer.getvalue())

## Étape 3 — Niveaux de sévérité

In [ ]:
NIVEAUX = {"DEBUG": 0, "INFO": 1, "AVERT": 2, "ERREUR": 3}

def enregistrer_niveau(buf, niveau, ligne, seuil="INFO"):
    """N'écrit que si niveau >= seuil."""/
    if NIVEAUX.get(niveau, 0) >= NIVEAUX.get(seuil, 0):
        horodatage = datetime.now().strftime("%H:%M:%S")
        buf.write(f"[{horodatage}][{niveau}] {ligne}\n")

demarrer_log(log_buffer)
enregistrer_niveau(log_buffer, "DEBUG",  "calcul interne ok",       seuil="INFO")   # filtré
enregistrer_niveau(log_buffer, "INFO",   "Tour 1 : drone_1 déplacé", seuil="INFO")  # écrit
enregistrer_niveau(log_buffer, "AVERT",  "batterie < 20 %",          seuil="INFO")  # écrit
enregistrer_niveau(log_buffer, "ERREUR", "collision détectée !",      seuil="INFO")  # écrit
print(log_buffer.getvalue())

## Étape 4 — Lire et filtrer un log existant

In [ ]:
def lire_lignes_niveau(buf, niveau_filtre):
    """Retourne uniquement les lignes du niveau demandé."""/
    buf.seek(0)
    return [l for l in buf.readlines() if f"[{niveau_filtre}]" in l]

erreurs = lire_lignes_niveau(log_buffer, "ERREUR")
print("Lignes ERREUR trouvées :")
for e in erreurs:
    print(' ', e.strip())

## Étape 5 — Résumé / bilan de partie

In [ ]:
def generer_bilan(buf):
    buf.seek(0)
    lignes = buf.readlines()
    total = sum(1 for l in lignes if l.startswith('['))  # lignes d'événements
    erreurs = sum(1 for l in lignes if '[ERREUR]' in l)
    avertissements = sum(1 for l in lignes if '[AVERT]' in l)
    print(f"Bilan : {total} événements — {erreurs} erreur(s) — {avertissements} avertissement(s)")/

generer_bilan(log_buffer)